# Agents, rewards and MDPs
**Machine Learning · Week 7 (Reinforcement learning) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w7d1.html

**Reinforcement learning:** an agent acts, the world answers with a reward and a new state, repeat.
Our world is the **cat world**, a 6 x 6 grid with a fish, a puddle and walls. Written down with its
rules, it is a small **Markov decision process (MDP)**.

In [ ]:
import numpy as np

maze = ["S...#.", ".##.#.", "...P..", ".#.##.", ".#....", "...#.F"]
rows, cols = len(maze), len(maze[0])
moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # the actions: up, right, down, left
names = ["up", "right", "down", "left"]

def outcomes(state, action, slip=0.0):
    """The MDP's rules: every possible (probability, next state, reward, episode over?)."""
    results = []
    for direction, p in [(action, 1 - slip), ((action + 1) % 4, slip / 2), ((action + 3) % 4, slip / 2)]:
        if p == 0:
            continue
        r, c = state
        nr, nc = r + moves[direction][0], c + moves[direction][1]
        if not (0 <= nr < rows and 0 <= nc < cols) or maze[nr][nc] == "#":
            nr, nc = r, c                             # bumped into a wall: stay put
        cell = maze[nr][nc]
        reward = 1.0 if cell == "F" else -1.0 if cell == "P" else -0.04
        results.append((p, (nr, nc), reward, cell == "F"))
    return results

def step(state, action, rng, slip=0.0):
    """Sample one move: returns (next state, reward, episode over?)."""
    results = outcomes(state, action, slip)
    i = rng.choice(len(results), p=[p for p, *_ in results])
    return results[i][1:]

print("\n".join(maze))
print("\nFrom the start, trying to go right on a slippery floor:")
for p, s2, r, done in outcomes((0, 0), 1, slip=0.2):
    print(f"   with probability {p:.1f}: land on {s2}, reward {r:+.2f}")

## One episode with a random policy
A **policy** chooses an action in every state. The simplest one picks at random. The **return** adds up
the rewards, each discounted by $\gamma$ per step of waiting: $G = r_1 + \gamma r_2 + \gamma^2 r_3 + \dots$

In [ ]:
def run_episode(policy, rng, gamma=0.9, slip=0.0, max_steps=100):
    state, G, rewards = (0, 0), 0.0, []
    for t in range(max_steps):
        action = policy(state, rng)
        state, reward, done = step(state, action, rng, slip)
        G += gamma ** t * reward
        rewards.append(reward)
        if done:
            break
    return G, rewards

random_policy = lambda state, rng: int(rng.integers(4))
rng = np.random.default_rng(3)
G, rewards = run_episode(random_policy, rng)
print(f"random policy: {len(rewards)} steps, {rewards.count(-1.0)} puddle visits, "
      f"{'reached the fish' if rewards[-1] == 1.0 else 'no fish within 100 steps'}")
print(f"return with gamma = 0.9: {G:.3f}")

## A policy written by hand, and its value
A table of arrows: one action for every square. Averaging the return over many episodes estimates the
**value** of the start square under that policy.

In [ ]:
arrows = ["DLLL#D",
          "D##U#D",
          "RRDRRD",
          "U#D##D",
          "U#RRRD",
          "URU#RF"]
act = {"U": 0, "R": 1, "D": 2, "L": 3}
hand_policy = lambda state, rng: act[arrows[state[0]][state[1]]]

for slip in [0.0, 0.2]:
    for name, policy in [("random", random_policy), ("hand-made", hand_policy)]:
        rng = np.random.default_rng(0)
        returns = [run_episode(policy, rng, slip=slip)[0] for _ in range(500)]
        print(f"slip {slip:.1f}, {name:9s} policy: average return {np.mean(returns):+.3f}")

## Your turn: what does the discount decide?
Write `discounted_return(rewards, gamma)`. Then compare two choices: a **quick** small fish (+0.5 after
3 steps) and a **patient** big fish (+1 after 12 steps). Which does each $\gamma$ prefer?

In [ ]:
def discounted_return(rewards, gamma):
    return 0.0          # TODO: add up gamma**k * rewards[k]

quick = [-0.04, -0.04, 0.5]
patient = [-0.04] * 11 + [1.0]
for gamma in [0.5, 0.8, 0.9, 0.95, 0.99]:
    print(gamma, discounted_return(quick, gamma), discounted_return(patient, gamma))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def discounted_return(rewards, gamma):
    return sum(gamma ** k * r for k, r in enumerate(rewards))

quick = [-0.04, -0.04, 0.5]
patient = [-0.04] * 11 + [1.0]
for gamma in [0.5, 0.8, 0.9, 0.95, 0.99]:
    q, p = discounted_return(quick, gamma), discounted_return(patient, gamma)
    print(f"gamma = {gamma:.2f}: quick {q:+.3f}, patient {p:+.3f} -> prefers the {'quick' if q > p else 'patient'} fish")
print("A small gamma makes the agent short-sighted; only a gamma close to 1 makes the long wait worth it.")